# Data Matching

In [1]:
import pandas as pd
from splink.datasets import splink_datasets
from splink import DuckDBAPI, Linker, SettingsCreator
import splink.comparison_library as cl
import splink
import inspect
from splink import Linker

print("Splink version:", splink.__version__)
print("Linker signature:")
print(inspect.signature(Linker))

Splink version: 5.0.0
Linker signature:
(splink_dataframe_or_dataframes: 'SplinkDataFrame | Sequence[SplinkDataFrame]', settings: 'SettingsCreator | dict[str, Any] | Path | str', log_level: 'int | str | None' = 20, validate_settings: 'bool' = True)


## Loade data

Kolonnebeskrivelser

• unique_id (eller tilsvarende id-kolonne):
	• En unik identifikator for hver enkelt række (transaktion) i datasættet. Denne bruges af SPlink som den primære nøgle (unique_id_column_name i indstillingerne) for at holde styr på, hvilke par der sammenlignes.
• transaction_date:
	• Datoen for transaktionen. I ERP- og bankafstemning er datoer fantastiske til record linkage, men de har ofte mindre tidsforskydninger (f.eks. hvis en transaktion sendes den 23. men bogføres den 24.). Du kan teste SPlinks evne til at lave "abs_diff" (absolut forskel i dage) på denne kolonne.
• amount:
	• Det finansielle beløb for transaktionen. Beløbet er ofte det stærkeste match-kriterium i ERP-afstemning, men værdierne har stærk dataskævhed (skew) – f.eks. mange overførsler på præcis samme runde beløb. Her kan du teste numeriske sammenligninger med procentvise eller absolutte tolerancer.
• memo:
	• Transaktionsteksten eller beskrivelsen (f.eks. "Faktura 10234", "Løn", "Overførsel"). Det er her, SPlink for alvor viser sit værd. I modtager-datasættet (transactions_destination) vil du opleve, at denne tekst ofte er afkortet (truncated), stavet forkert eller helt mangler. Det er den perfekte kolonne til at teste Jaro-Winkler eller Levenshtein tekst-sammenligninger.

In [2]:
# To store transaktionsdatasæt (perfekt til ERP-/finanstest)
df_origin = splink_datasets.transactions_origin.to_pandas()
df_destination = splink_datasets.transactions_destination.to_pandas()

print(df_origin.head())
print(f"Antal rækker i oprindelig tabel: {len(df_origin)}")

   ground_truth             memo transaction_date  amount  unique_id
0             0  MATTHIAS C paym       2022-03-28   36.36          0
1             1  M CORVINUS dona       2022-02-14  221.91          1
2             2  M C donation BG       2022-05-04  449.79          2
3             3          M C BGC       2022-03-03  208.45          3
4             4  M CORVINUS  CSH       2022-02-04   79.74          4
Antal rækker i oprindelig tabel: 45326


### Understand the destination dataset

In [3]:
print("Rows:", len(df_destination))
print("\nData types:")
print(df_destination.dtypes)

print("\nMissing values:")
display(df_destination.isna().sum())

print("\nUnique values:")
display(df_destination.nunique())

print("\nSample:")
display(df_destination)

Rows: 45326

Data types:
ground_truth          int64
memo                 object
transaction_date     object
amount              float64
unique_id             int64
dtype: object

Missing values:


ground_truth        0
memo                0
transaction_date    0
amount              0
unique_id           0
dtype: int64


Unique values:


ground_truth        45326
memo                36010
transaction_date      149
amount              34187
unique_id           45326
dtype: int64


Sample:


,ground_truth,memo,transaction_date,amount,unique_id
0,0,MATTHIAS C payment BGC,2022-03-29,36.36,0
1,1,M CORVINUS BGC,2022-02-16,221.91,1
2,2,M C,2022-05-05,449.79,2
3,3,M C payment,2022-03-04,198.79,3
4,4,M CORVINUS,2022-02-05,79.74,4
...,...,...,...,...,...
45321,45321,L STRONG JUDD p,2022-03-21,48413.25,45321
45322,45322,LEWIS S JUDD money,2022-03-04,19016.96,45322
45323,45323,L STRONG JUDD money BGC,2022-03-10,823.81,45323
45324,45324,L S JUDD donati,2022-03-05,81644.55,45324


## Find duplicates
First we are going to find all exact duplicated rows. Our search for duplicated rows will be focused on the columns "memo", "transaction_date" and "amount".

In [4]:
match_columns = [
    "memo",
    "transaction_date",
    "amount"
]

exact_duplicates = df_destination = splink_datasets.transactions_destination.to_pandas()
exact_duplicates = df_destination[
    df_destination.duplicated(
        subset=match_columns,
        keep=False
    )
].sort_values(match_columns)

print(
    "Rows in exact duplicate groups:",
    len(exact_duplicates)
)

display(exact_duplicates.head(20))

Rows in exact duplicate groups: 0


,ground_truth,memo,transaction_date,amount,unique_id


Our search for duplicated rows did not find any matches.

## Investigating Potential Duplicates

When working with transactional data, identifying duplicates is not always as straightforward as finding identical rows. Two records may represent the same transaction even when some of their attributes differ due to inconsistent descriptions, formatting differences, or data entry errors.

Before introducing probabilistic record linkage with Splink, we first investigate potential duplicates by examining transactions that share the same date and amount but have different descriptions (memo).

These records are not necessarily duplicates. Multiple legitimate transactions can occur on the same date for the same amount. However, identifying such groups helps us understand the data and determine which attributes may be useful for matching.

This exploratory data analysis (EDA) will help us design more effective comparison and blocking rules for our Splink model, rather than relying on assumptions about the dataset.

In [5]:
candidate_groups = (
    df_destination.groupby(["transaction_date", "amount"])
      .agg(
          records=("unique_id", "count"),
          unique_memos=("memo", "nunique")
      )
      .reset_index()
)

candidates = candidate_groups.query(
    "records > 1 and unique_memos > 1"
)

display(
    candidates.sort_values(
        "records",
        ascending=False
    ).head(20)
)

,transaction_date,amount,records,unique_memos
348,2022-01-14,79.39,2,2
536,2022-01-17,42.05,2,2
960,2022-01-22,110.25,2,2
1941,2022-01-30,123.51,2,2
3024,2022-02-06,36.98,2,2
3027,2022-02-06,43.64,2,2
4291,2022-02-12,324.92,2,2
5127,2022-02-16,100.42,2,2
5332,2022-02-17,45.85,2,2
5573,2022-02-18,34.80,2,2


## Build Splink Model

The exploratory data analysis (EDA) revealed that some transactions share the same date and amount while having different descriptions. Although this does not necessarily mean that these records are duplicates, it suggests that further investigation is needed.

To identify potential duplicates, we will use **Splink**, a Python library for probabilistic record linkage and entity resolution.

Unlike traditional duplicate detection, which relies on exact matches, Splink evaluates similarities between multiple attributes and estimates the probability that two records represent the same entity or transaction.

For our initial model, we will compare three attributes:

- **Memo:** Using Jaro-Winkler similarity to identify transaction descriptions that are similar but not identical.
- **Amount:** Using exact matching to determine whether transactions have the same amount.
- **Transaction date:** Using exact matching to identify transactions recorded on the same date.

We will also introduce **blocking rules** to reduce the number of record comparisons. Instead of comparing every transaction with every other transaction, Splink will focus on records that share selected attributes.

Since our first objective is to identify duplicates within `df_destination`, we configure Splink with `link_type="dedupe_only"`.

This initial configuration provides a starting point for probabilistic duplicate detection. However, a high match probability does not automatically confirm that two transactions are duplicates. We will need to examine the results and evaluate whether our comparison methods and blocking rules are appropriate for the dataset.

In [6]:
settings = SettingsCreator(
    link_type="dedupe_only",
    unique_id_column_name="unique_id",

    comparisons=[
        cl.JaroWinklerAtThresholds(
            "memo",
            [0.95, 0.90, 0.80]
        ),

        cl.ExactMatch("amount"),

        cl.ExactMatch("transaction_date")
    ],

    blocking_rules_to_generate_predictions=[
        "l.transaction_date = r.transaction_date "
        "AND l.amount = r.amount"
    ],

    retain_intermediate_calculation_columns=True
)

db_api = DuckDBAPI()

df_splink = db_api.register(
    df_destination,
    dataset_display_name="destination"
)

linker = Linker(
    df_splink,
    settings
)

print("Splink model initialized successfully!")

Splink model initialized successfully!


In [7]:
print(inspect.signature(Linker.__init__))

(self, splink_dataframe_or_dataframes: 'SplinkDataFrame | Sequence[SplinkDataFrame]', settings: 'SettingsCreator | dict[str, Any] | Path | str', log_level: 'int | str | None' = 20, validate_settings: 'bool' = True)


## Train the model

In [8]:
linker.training.estimate_u_using_random_sampling(
    max_pairs=1e6
)

linker.training.estimate_probability_two_random_records_match(
    [
        "l.transaction_date = r.transaction_date "
        "AND l.amount = r.amount"
    ],
    recall=0.7
)

linker.training.estimate_parameters_using_expectation_maximisation(
    "l.amount = r.amount"
)

You are using the default value for `max_pairs`, which may be too small and thus lead to inaccurate estimates for your model's u-parameters. Consider increasing to 1e8 or 1e9, which will result in more accurate estimates, but with a longer run time.
----- Estimating u probabilities using random sampling -----
Estimating u with: max_pairs = 1,000,000, min_count_per_level = 100, num_chunks = 10

Estimating u for: memo (Comparison 1 of 3)
  Running probe chunk (~1.00% of max_pairs)
  Min u_count: 0 for comparison level Jaro-Winkler distance of memo >= 0.95 (cvv=3)
  Probe did not converge; restarting with normal chunking

  Running chunk 1/10
  Count of 0 for level Exact match on memo (cvv=4). Chunk took 0.1 seconds.
  Min u_count not hit, continuing.
  Running chunk 2/10
  Count of 0 for level Exact match on memo (cvv=4). Chunk took 0.1 seconds.
  Min u_count not hit, continuing.
  Running chunk 3/10
  Count of 0 for level Exact match on memo (cvv=4). Chunk took 0.1 seconds.
  Min u_coun

<EMTrainingSession, blocking on l.amount = r.amount, deactivating comparisons amount>

## Find duplicates

In [9]:
predictions = linker.inference.predict(
    threshold_match_probability=0.7
)

matches = predictions.as_pandas_dataframe()

print("Number of potential matches:", len(matches))

print("\nAvailable columns:")
print(matches.columns.tolist())

columns = [
    "unique_id_l",
    "unique_id_r",
    "match_probability"
]

display(
    matches[columns]
    .sort_values(
        "match_probability",
        ascending=False
    )
    .head(20)
)

Blocking time: 0.03 seconds
Predict time (post-blocking): 0.07 seconds

 -- WARNING --
You have called predict(), but there are some parameter estimates which have neither been estimated or specified in your settings dictionary.  To produce predictions the following untrained parameters will use default values.
Comparison: 'amount':
    m values not fully trained


Number of potential matches: 1

Available columns:
['match_weight', 'match_probability', 'unique_id_l', 'unique_id_r', 'memo_l', 'memo_r', 'gamma_memo', 'mw_memo', 'amount_l', 'amount_r', 'gamma_amount', 'mw_amount', 'transaction_date_l', 'transaction_date_r', 'gamma_transaction_date', 'mw_transaction_date', 'match_key']


,unique_id_l,unique_id_r,match_probability
0,6780,6795,0.995365


## Visualize matchresults

In [10]:
linker.visualisations.match_weights_chart()

SplinkChart (use .to_html() or .save() to export HTML)